# Lab 1: Adult Dataset EDA and Statistical Analysis

## Objective
This notebook solves the Lab 1 assignment using the UCI Adult dataset with code and explanations.

## Hypothesis
The dataset supports binary classification for income prediction: whether a person earns more than \$50K per year.

## What We Will Extract
- Basic data understanding (shape, types, missing values, unique classes)
- Distribution behavior of important numeric columns
- Categorical feature patterns related to income
- One-hot encoded feature matrix for future ML work
- Required statistical metrics and visual comparisons for male/female populations

## Environment Setup
This cell imports required libraries. If `pandas` or `seaborn` are missing, it installs them automatically.

In [ ]:
import importlib.util
import subprocess
import sys

required_pkgs = ["pandas", "seaborn"]
for pkg in required_pkgs:
    if importlib.util.find_spec(pkg) is None:
        print(f"Installing missing package: {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 80)
sns.set_theme(style="whitegrid")

print("Setup complete.")


## Load Dataset and Define Schema
We load the dataset from the official UCI source and assign explicit column names.

In [ ]:
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

column_names = [
    "Age", "WorkClass", "fnlwgt", "Education", "EducationNum", "MaritalStatus",
    "Occupation", "Relationship", "Race", "Gender", "CapitalGain", "CapitalLoss",
    "HoursPerWeek", "NativeCountry", "Income"
]

df_raw = pd.read_csv(DATA_URL, header=None, names=column_names, skipinitialspace=True)

# Trim whitespace in all text columns (works cleanly across pandas 2/3)
for col in df_raw.select_dtypes(include=["object", "string"]).columns:
    df_raw[col] = df_raw[col].str.strip()

print("Loaded dataset successfully")
print("Shape:", df_raw.shape)
print("Columns:", list(df_raw.columns))


## Initial EDA: Head, Tail, Shape, Data Types

In [ ]:
print("Head:")
display(df_raw.head())

print("Tail:")
display(df_raw.tail())

print("Shape:", df_raw.shape)
print()
print("Data types:")
print(df_raw.dtypes)

In [ ]:
print("NaN counts per column (before handling '?' tokens):")
print(df_raw.isna().sum())

print()
print("Unique values in Income:")
print(df_raw["Income"].unique())

income_map = {"<=50K": -1, ">50K": 1}
df_raw["IncomeBinary"] = df_raw["Income"].map(income_map)

if df_raw["IncomeBinary"].isna().any():
    bad_values = sorted(df_raw.loc[df_raw["IncomeBinary"].isna(), "Income"].unique())
    raise ValueError(f"Unexpected Income values found: {bad_values}")

y = df_raw["IncomeBinary"].to_numpy(dtype=int)
X_df = df_raw.drop(columns=["Income", "IncomeBinary"]).copy()

print()
print("Income mapping completed.")
print("Unique target labels in y:", np.unique(y))

`y` is now the binary target array with values `{-1, +1}` and `X_df` contains only features.

In [ ]:
print("Feature dataframe (X_df) head:")
display(X_df.head())

print("Summary statistics for numeric columns:")
display(X_df.describe())

## Sparse-Value Analysis for CapitalGain and CapitalLoss
The assignment asks whether these are heavily concentrated at zero.

In [ ]:
capital_gain_zero_pct = (df_raw["CapitalGain"] == 0).mean() * 100
capital_loss_zero_pct = (df_raw["CapitalLoss"] == 0).mean() * 100

print(f"CapitalGain % zeros: {capital_gain_zero_pct:.2f}%")
print(f"CapitalLoss % zeros: {capital_loss_zero_pct:.2f}%")

Both columns are highly sparse in this dataset (large mass at zero), which can create wide but mostly-empty feature representations.
As requested, we drop both columns before later encoding.

In [ ]:
df_sparse_dropped = df_raw.drop(columns=["CapitalGain", "CapitalLoss", "IncomeBinary"]).copy()

print("Shape after dropping CapitalGain and CapitalLoss:", df_sparse_dropped.shape)
print("Remaining columns:")
print(df_sparse_dropped.columns.tolist())

## Categorical Variable Exploration
Required categorical columns:
- WorkClass, Education, MaritalStatus, Occupation, Relationship, Race, Gender, NativeCountry

In [ ]:
categorical_cols = [
    "WorkClass", "Education", "MaritalStatus", "Occupation",
    "Relationship", "Race", "Gender", "NativeCountry"
]

for col in categorical_cols:
    print()
    print(f"{col} unique values:")
    print(sorted(df_sparse_dropped[col].dropna().unique().tolist()))

In [ ]:
df_clean = df_sparse_dropped.copy()

# Convert '?' placeholders to missing values, then drop rows containing missing categorical values.
df_clean[categorical_cols] = df_clean[categorical_cols].replace("?", np.nan)

rows_before = len(df_clean)
missing_before_drop = df_clean[categorical_cols].isna().sum().sort_values(ascending=False)

df_clean = df_clean.dropna(subset=categorical_cols).copy()
rows_after = len(df_clean)

print("Missing counts in categorical columns before drop:")
print(missing_before_drop)
print()
print("Rows before drop:", rows_before)
print("Rows after drop:", rows_after)
print("Rows removed:", rows_before - rows_after)

In [ ]:
distinct_counts = (
    df_clean[categorical_cols]
    .nunique()
    .rename("DistinctCount")
    .sort_values(ascending=False)
    .to_frame()
)

print("Distinct count of each categorical column:")
display(distinct_counts)

In [ ]:
# Category-level income tendency (share of >50K) for each categorical feature.
# Use a minimum category size filter to avoid very noisy tiny groups.
MIN_GROUP_SIZE = 50

for col in categorical_cols:
    temp = (
        df_clean.groupby(col)["Income"]
        .agg(
            Count="count",
            HighIncomeRate=lambda s: (s == ">50K").mean() * 100,
        )
        .sort_values("HighIncomeRate", ascending=False)
    )
    temp = temp[temp["Count"] >= MIN_GROUP_SIZE]

    print()
    print(f"{col} - top categories by >50K rate (Count >= {MIN_GROUP_SIZE}):")
    display(temp.head(8))

### Categorical Analysis Notes (w.r.t. Hypothesis)
- `Education` and `Occupation` usually show strong separation in the `>50K` rate.
- `MaritalStatus` and `Relationship` often have noticeable income distribution differences.
- `Gender` comparison is central to the assignment and is analyzed again in dedicated metric/plot sections.
- `NativeCountry` can have many levels; larger groups are more reliable than tiny groups.

## One-Hot Encoding
Now we encode all categorical columns for model-ready numerical features.

In [ ]:
features_for_encoding = df_clean.drop(columns=["Income"]).copy()

encoded_df = pd.get_dummies(
    features_for_encoding,
    columns=categorical_cols,
    drop_first=False,
    dtype=int,
)

print("Encoded dataframe shape:", encoded_df.shape)
print("Preview:")
display(encoded_df.head())

## Required Metrics (Male/Female and Income)
The following block computes all required assignment statistics.

In [ ]:
male_df = df_clean[df_clean["Gender"] == "Male"].copy()
female_df = df_clean[df_clean["Gender"] == "Female"].copy()

male_total = len(male_df)
female_total = len(female_df)

male_gt_50k = (male_df["Income"] == ">50K").sum()
female_gt_50k = (female_df["Income"] == ">50K").sum()

avg_age_male_gt_50k = male_df.loc[male_df["Income"] == ">50K", "Age"].mean()
avg_age_female_le_50k = female_df.loc[female_df["Income"] == "<=50K", "Age"].mean()

std_male_age = male_df["Age"].std()
std_female_age = female_df["Age"].std()

skew_male_age = male_df["Age"].skew()
skew_female_age = female_df["Age"].skew()

mean_age_diff = male_df["Age"].mean() - female_df["Age"].mean()

metrics = pd.DataFrame(
    {
        "Metric": [
            "Total male count",
            "Total female count",
            "Male count with income >50K",
            "Female count with income >50K",
            "Average age of males with income >50K",
            "Average age of females with income <=50K",
            "Std. deviation of male age",
            "Std. deviation of female age",
            "Skewness of male age",
            "Skewness of female age",
            "Mean age difference (male - female)",
        ],
        "Value": [
            male_total,
            female_total,
            male_gt_50k,
            female_gt_50k,
            avg_age_male_gt_50k,
            avg_age_female_le_50k,
            std_male_age,
            std_female_age,
            skew_male_age,
            skew_female_age,
            mean_age_diff,
        ],
    }
)

print("Required metrics:")
display(metrics)

The table above directly answers the numeric checklist from the assignment.

## Visual Analysis
We now produce the required and supporting plots.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_clean["Age"], bins=30, kde=True, ax=axes[0], color="#4C78A8")
axes[0].set_title("Age Distribution")
axes[0].set_xlabel("Age")

sns.histplot(df_clean["HoursPerWeek"], bins=30, kde=True, ax=axes[1], color="#F58518")
axes[1].set_title("HoursPerWeek Distribution")
axes[1].set_xlabel("HoursPerWeek")

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=df_clean, x="Income", ax=axes[0], palette="Set2")
axes[0].set_title("Income Class Count")

sns.countplot(data=df_clean, x="Gender", ax=axes[1], palette="Set1")
axes[1].set_title("Gender Count")

plt.tight_layout()
plt.show()

In [ ]:
high_income_df = df_clean[df_clean["Income"] == ">50K"].copy()

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=high_income_df,
    x="Age",
    y="HoursPerWeek",
    hue="Gender",
    alpha=0.65,
)
plt.title("Scatter Plot (Income >50K): Age vs HoursPerWeek by Gender")
plt.xlabel("Age")
plt.ylabel("HoursPerWeek")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_clean, x="Gender", y="Age", hue="Income", palette="pastel")
plt.title("Box Plot: Age Distribution by Gender and Income")
plt.xlabel("Gender")
plt.ylabel("Age")
plt.tight_layout()
plt.show()

In [ ]:
age_bins = [16, 25, 35, 45, 55, 65, 100]
age_labels = ["17-25", "26-35", "36-45", "46-55", "56-65", "66+"]

line_df = df_clean.copy()
line_df["AgeBin"] = pd.cut(
    line_df["Age"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True,
)

income_rate_by_age_gender = (
    line_df.groupby(["AgeBin", "Gender"], observed=False)["Income"]
    .apply(lambda s: (s == ">50K").mean() * 100)
    .reset_index(name="HighIncomeRatePct")
)

plt.figure(figsize=(10, 6))
sns.lineplot(
    data=income_rate_by_age_gender,
    x="AgeBin",
    y="HighIncomeRatePct",
    hue="Gender",
    marker="o",
)
plt.title("Income >50K Rate by Age Group and Gender")
plt.xlabel("Age Group")
plt.ylabel(">50K Rate (%)")
plt.tight_layout()
plt.show()

## Conclusion
- The notebook confirms the Adult dataset is suitable for binary income analysis (`<=50K` vs `>50K`).
- We completed full EDA, handled sparse and missing categorical values, and prepared one-hot encoded features.
- Required male/female counts, means, standard deviations, skewness, and mean age difference were computed.
- Visualizations highlight distribution and income-pattern differences by gender and age.

### Limitations
- This is observational census data; associations do not imply causation.
- Rows with `?` in categorical variables were dropped, which can shift distributions.
- No model training is included here by design; the notebook is now ready for the machine-learning phase.